# Cluster-Coupled Adam 변형 실험 (코랩 GPU)

1. 메뉴 **런타임 → 런타임 유형 변경 → GPU(T4)** 를 고른다.
2. 위에서부터 셀을 차례로 실행한다. 결과 CSV는 구글 드라이브 `MyDrive/cca_experiments/`에 저장되고,
   이미 끝난 실행은 건너뛰므로 런타임이 끊겨도 다시 실행하면 이어서 돈다.
3. 마지막 셀이 방법별 평균과 Adam 대비 변화율 표를 만든다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/cca_experiments'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf cluster-coupled-adam && git clone -q https://github.com/sjzzang776/cluster-coupled-adam.git
!pip -q install tensorboardX==2.6
%cd /content/cluster-coupled-adam/code

## 데이터
Gowalla는 저장소에 들어 있다. Yelp2018·Amazon-Book은 LightGCN 원 저장소(gusye1234/LightGCN-PyTorch)에서 받는다.

In [ ]:
import urllib.request, os
BASE = 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data'
for ds in ['yelp2018', 'amazon-book']:
    os.makedirs(f'../data/{ds}', exist_ok=True)
    for f in ['train.txt', 'test.txt']:
        dst = f'../data/{ds}/{f}'
        if not os.path.exists(dst):
            urllib.request.urlretrieve(f'{BASE}/{ds}/{f}', dst)
    print(ds, 'ok')

## 실험 목록
한 줄 = (이름, 데이터셋, 시드, 추가 인자). 필요한 줄만 남기고 지워도 된다.

In [ ]:
COMMON = '--model lgn --recdim 32 --layer 1 --bpr_batch 4096 --testbatch 512 --num_clusters 64 --recluster_interval 128 --cluster_warmup 1024 --tensorboard 0'
VARIANTS = {
    'adam':     '--optimizer adam',
    'cca':      '--optimizer cluster --alpha 0.25',
    'vshare':   '--optimizer cluster --alpha 1.0 --mix_mode vshare',
    'vwarm':    '--optimizer cluster --alpha 1.0 --mix_mode vshare --alpha_n0 20 --alpha_freq_dir warm',
    'shrink':   '--optimizer cluster --alpha 1.0 --mix_mode shrink',
}
EPOCHS = 30
JOBS = [(v, ds, seed) for ds in ['gowalla', 'yelp2018'] for seed in [2020, 2021, 2022] for v in VARIANTS]
print(len(JOBS), 'runs')

In [ ]:
import subprocess, glob, shutil, time
for v, ds, seed in JOBS:
    tag = f'{ds}_{v}_e{EPOCHS}_s{seed}'
    if os.path.exists(f'{OUT}/{tag}.csv'):
        print('skip', tag); continue
    for f in glob.glob('runs/*.csv'): os.remove(f)
    t = time.time()
    cmd = f'python -u main.py --dataset {ds} --epochs {EPOCHS} --seed {seed} {COMMON} {VARIANTS[v]}'
    with open(f'{OUT}/{tag}.log', 'w') as log:
        code = subprocess.call(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
    csvs = sorted(glob.glob('runs/*.csv'), key=os.path.getmtime)
    if csvs: shutil.copy(csvs[-1], f'{OUT}/{tag}.csv')
    print(f'{tag} exit={code} {(time.time()-t)/60:.1f}min')

## 결과 표 (마지막 에폭, 시드 평균 ± 표준편차, Adam 대비 같은 시드끼리 변화율)

In [ ]:
import pandas as pd
rows = []
for f in glob.glob(f'{OUT}/*.csv'):
    ds, v, ep, s = os.path.basename(f)[:-4].rsplit('_', 3)
    r = pd.read_csv(f).iloc[-1]
    rows.append(dict(dataset=ds, variant=v, seed=s, **{m: r[m] for m in ['recall@20', 'ndcg@20', 'hr@20', 'recall_head', 'recall_tail']}))
df = pd.DataFrame(rows)
M = ['recall@20', 'ndcg@20', 'hr@20', 'recall_head', 'recall_tail']
display(df.groupby(['dataset', 'variant'])[M].agg(['mean', 'std']).round(4))
base = df[df.variant == 'adam'].set_index(['dataset', 'seed'])[M]
rel = df.set_index(['dataset', 'seed']).join(base, rsuffix='_adam')
for m in M: rel[m + ' vs adam %'] = (rel[m] / rel[m + '_adam'] - 1) * 100
display(rel.groupby(['dataset', 'variant'])[[m + ' vs adam %' for m in M]].mean().round(2))